# Diffusion Models: Denoising Diffusion Probabilistic Models (DDPM)

**Learning Objectives:**
- Understand the forward diffusion process (adding noise)
- Learn the reverse diffusion process (learning to denoise)
- Implement U-Net architecture with time embeddings
- Compare sampling strategies (DDPM vs DDIM)
- Connect diffusion models to score-based models
- Generate high-quality images from noise

**What We'll Build:**
1. Forward diffusion: gradually add noise to images
2. U-Net with time embeddings: predict noise at each timestep
3. DDPM training: learn to reverse the diffusion process
4. DDIM sampling: faster inference with deterministic sampling
5. Image generation from pure noise

**Why Diffusion Models Matter:**
- Power Stable Diffusion, DALL-E 2, Midjourney
- State-of-the-art image generation quality
- More stable training than GANs
- Better mode coverage than VAEs

## 1. Introduction: What Are Diffusion Models?

**Core Idea:**
- **Forward process**: Gradually add noise to data until it becomes pure noise
- **Reverse process**: Learn to remove noise step-by-step, generating data from noise

**Intuition:**
Imagine watching a photograph fade into static noise, then learning to reverse this process. If you can reverse the fading perfectly, you can generate new photographs from pure static!

**Comparison to other generative models:**

| Model | How it works | Strengths | Weaknesses |
|-------|--------------|-----------|------------|
| **VAE** | Encode → latent → decode | Stable training, fast sampling | Blurry outputs |
| **GAN** | Generator vs discriminator | Sharp images | Unstable training, mode collapse |
| **Diffusion** | Iterative denoising | Sharp images, stable training, mode coverage | Slow sampling (many steps) |

**Key Innovation:**
Instead of generating images in one shot, diffusion models generate images through **many small denoising steps** (e.g., 1000 steps). Each step is easier to learn than the full transformation.

## 2. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import math

from aiml_notebooks import set_seed, get_device

# Set random seed for reproducibility
set_seed(42)

# Get device
device = get_device()
print(f"Using device: {device}")

## 3. Load MNIST Dataset

We'll use MNIST for faster training and clearer visualization of concepts.

In [ ]:
# Transform: normalize to [-1, 1] range (common for diffusion models)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Scale to [-1, 1]
])

# Load dataset
train_dataset = datasets.MNIST(
    root='./tmp/data',
    train=True,
    download=True,
    transform=transform
)

# Create dataloader
batch_size = 128
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

print(f"Dataset size: {len(train_dataset)}")
print(f"Batches: {len(train_loader)}")
print(f"Image shape: {train_dataset[0][0].shape}")

Visualize sample images from the dataset.

In [ ]:
# Visualize samples
def show_images(images, title="", nrow=8):
    """Display a grid of images."""
    # Denormalize from [-1, 1] to [0, 1]
    images = (images + 1) / 2
    images = torch.clamp(images, 0, 1)
    
    grid = make_grid(images, nrow=nrow, padding=2)
    plt.figure(figsize=(12, 6))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Get sample batch
sample_images, _ = next(iter(train_loader))
show_images(sample_images[:64], "Original MNIST Images")

## 4. Forward Diffusion Process - Theory

The forward process gradually adds **Gaussian noise** to images over $T$ timesteps.

### Mathematical Formulation:

At each timestep $t \in \{1, ..., T\}$, we add noise according to:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1 - \beta_t} x_{t-1}, \beta_t I)$$

Where:
- $x_0$ is the original image
- $x_t$ is the noisy image at timestep $t$
- $\beta_t$ is the **noise schedule** (how much noise to add at step $t$)

### Key Property: Closed-Form Sampling

We can jump directly to any timestep $t$ without computing all intermediate steps!

Let $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$, then:

$$q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t} x_0, (1 - \bar{\alpha}_t) I)$$

Or equivalently:
$$x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

**Intuition:**
- $\sqrt{\bar{\alpha}_t}$: How much of the original image remains
- $\sqrt{1 - \bar{\alpha}_t}$: How much noise has been added
- As $t \to T$: $\bar{\alpha}_t \to 0$, so $x_T \approx \epsilon$ (pure noise)

## 5. Implementing Forward Diffusion

Let's implement the noise schedule and forward process.

In [ ]:
class DiffusionSchedule:
    """
    Manages the noise schedule for diffusion models.
    
    The schedule defines beta_t values that control how much noise
    is added at each timestep.
    """
    
    def __init__(self, timesteps=1000, beta_start=0.0001, beta_end=0.02, device='cpu'):
        """
        Args:
            timesteps: Number of diffusion steps T
            beta_start: Starting noise level (small)
            beta_end: Ending noise level (larger)
            device: Device to store tensors
        """
        self.timesteps = timesteps
        self.device = device
        
        # Linear schedule: beta increases linearly from beta_start to beta_end
        self.betas = torch.linspace(beta_start, beta_end, timesteps, device=device)
        
        # Compute alpha values
        self.alphas = 1.0 - self.betas
        
        # Compute cumulative product of alphas (alpha_bar)
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        
        # Shifted version for computing posterior
        self.alphas_cumprod_prev = torch.cat([
            torch.tensor([1.0], device=device),
            self.alphas_cumprod[:-1]
        ])
        
        # Precompute useful values
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / self.alphas)
        
        # For posterior q(x_{t-1} | x_t, x_0)
        self.posterior_variance = (
            self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
    
    def get_index_from_list(self, vals, t, x_shape):
        """
        Extract values from vals at indices t and reshape for broadcasting.
        
        Args:
            vals: Tensor of values (length = timesteps)
            t: Timestep indices (batch_size,)
            x_shape: Shape of x for broadcasting
        
        Returns:
            Values at indices t, reshaped for broadcasting
        """
        batch_size = t.shape[0]
        out = vals.gather(-1, t)
        # Reshape to (batch_size, 1, 1, 1) for broadcasting with images
        return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))
    
    def forward_diffusion(self, x_0, t, noise=None):
        """
        Apply forward diffusion: q(x_t | x_0)
        
        Args:
            x_0: Original images (batch_size, C, H, W)
            t: Timesteps (batch_size,)
            noise: Optional pre-sampled noise (for reproducibility)
        
        Returns:
            x_t: Noisy images at timestep t
            noise: The noise that was added
        """
        if noise is None:
            noise = torch.randn_like(x_0)
        
        # Get sqrt(alpha_bar_t) and sqrt(1 - alpha_bar_t)
        sqrt_alpha_bar_t = self.get_index_from_list(
            self.sqrt_alphas_cumprod, t, x_0.shape
        )
        sqrt_one_minus_alpha_bar_t = self.get_index_from_list(
            self.sqrt_one_minus_alphas_cumprod, t, x_0.shape
        )
        
        # Apply: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise
        x_t = sqrt_alpha_bar_t * x_0 + sqrt_one_minus_alpha_bar_t * noise
        
        return x_t, noise


# Create diffusion schedule
timesteps = 1000
schedule = DiffusionSchedule(timesteps=timesteps, device=device)

print(f"Diffusion schedule created with {timesteps} timesteps")
print(f"Beta range: [{schedule.betas[0]:.6f}, {schedule.betas[-1]:.6f}]")
print(f"Alpha_bar at t=500: {schedule.alphas_cumprod[500]:.6f}")
print(f"Alpha_bar at t=999: {schedule.alphas_cumprod[999]:.6f}")

## 6. Visualizing Forward Diffusion

Let's see what happens when we add noise progressively to an image.

In [ ]:
# Take a single image
x_0 = sample_images[0:1].to(device)  # Shape: (1, 1, 28, 28)

# Visualize at different timesteps
timesteps_to_show = [0, 50, 100, 200, 400, 600, 800, 999]
noisy_images = []

for t in timesteps_to_show:
    if t == 0:
        noisy_images.append(x_0)
    else:
        t_tensor = torch.tensor([t], device=device)
        x_t, _ = schedule.forward_diffusion(x_0, t_tensor)
        noisy_images.append(x_t)

# Display
noisy_images = torch.cat(noisy_images, dim=0)
show_images(noisy_images, "Forward Diffusion: Adding Noise Over Time", nrow=8)

# Print noise levels
print("\nNoise levels (sqrt(1 - alpha_bar_t)):")
for t in timesteps_to_show:
    if t == 0:
        print(f"t={t:3d}: 0.0000 (original image)")
    else:
        noise_level = schedule.sqrt_one_minus_alphas_cumprod[t].item()
        signal_level = schedule.sqrt_alphas_cumprod[t].item()
        print(f"t={t:3d}: {noise_level:.4f} (signal: {signal_level:.4f})")

### Key Observations:

1. **Gradual degradation**: Image slowly becomes noise
2. **At t=999**: Almost pure Gaussian noise
3. **Signal preservation**: Early timesteps preserve most of the image
4. **Noise schedule**: Controls how quickly the image degrades

**Goal of reverse process**: Learn to go from right to left (noise → image)

## 7. Reverse Diffusion - Theory

The reverse process learns to **remove noise** step by step: $p_\theta(x_{t-1} | x_t)$

### Key Insight:

If we knew the true reverse distribution $q(x_{t-1} | x_t, x_0)$, we could reverse the process exactly. But we don't have $x_0$ during generation!

**Solution**: Train a neural network $\epsilon_\theta(x_t, t)$ to predict the noise $\epsilon$ that was added.

### Training Objective:

Simple MSE loss between predicted and actual noise:

$$L_{\text{simple}} = \mathbb{E}_{t, x_0, \epsilon} \left[ \|\epsilon - \epsilon_\theta(x_t, t)\|^2 \right]$$

Where:
- $t \sim \text{Uniform}(\{1, ..., T\})$: Random timestep
- $x_0 \sim q(x_0)$: Real image from dataset
- $\epsilon \sim \mathcal{N}(0, I)$: Random noise
- $x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon$: Forward process

### Sampling Algorithm (DDPM):

Start with $x_T \sim \mathcal{N}(0, I)$ (pure noise), then iterate:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right) + \sigma_t z$$

Where:
- $z \sim \mathcal{N}(0, I)$ for $t > 1$, else $z = 0$
- $\sigma_t = \sqrt{\beta_t}$ (noise variance)

**Intuition**: Each step removes some predicted noise and adds a small amount of random noise (except final step).

## 8. U-Net Architecture with Time Embeddings

The network $\epsilon_\theta(x_t, t)$ must:
1. Take noisy image $x_t$ as input
2. Take timestep $t$ as input (to know how much noise)
3. Output predicted noise $\epsilon_\theta$

**Architecture choice**: U-Net with skip connections and time embeddings

### Why U-Net?
- **Encoder-decoder**: Captures multi-scale features
- **Skip connections**: Preserves spatial information
- **Standard for diffusion**: Used in DDPM, Stable Diffusion, etc.

### Time Embeddings:
- Encode timestep $t$ into a high-dimensional vector
- Use **sinusoidal embeddings** (like Transformer positional encoding)
- Inject into each layer via addition or concatenation

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    """
    Sinusoidal time embeddings.
    
    Encodes timesteps using sine and cosine functions at different frequencies,
    similar to positional encodings in Transformers.
    """
    
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    
    def forward(self, t):
        """
        Args:
            t: Timesteps (batch_size,)
        
        Returns:
            Embeddings (batch_size, dim)
        """
        device = t.device
        half_dim = self.dim // 2
        
        # Compute frequencies
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        
        # Compute embeddings
        emb = t[:, None] * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        
        return emb


# Test time embeddings
time_emb = SinusoidalTimeEmbedding(128)
t = torch.tensor([0, 100, 500, 999])
emb = time_emb(t)

print(f"Time embedding shape: {emb.shape}")
print(f"Embeddings are different for different t: {not torch.allclose(emb[0], emb[1])}")

# Visualize embeddings
plt.figure(figsize=(12, 4))
plt.imshow(emb.detach().numpy(), aspect='auto', cmap='coolwarm')
plt.colorbar()
plt.xlabel('Embedding Dimension')
plt.ylabel('Timestep')
plt.title('Time Embeddings for Different Timesteps')
plt.yticks(range(4), ['t=0', 't=100', 't=500', 't=999'])
plt.tight_layout()
plt.show()

Now let's build the U-Net architecture with time conditioning.

In [ ]:
class ResidualBlock(nn.Module):
    """
    Residual block with time embedding injection.
    
    Architecture:
        x --> Conv --> GroupNorm --> Act --> Conv --> GroupNorm --> + --> out
        |                                                           |
        +-- (optional projection) ----------------------------------+
        
    Time embedding is added after the first activation.
    """
    
    def __init__(self, in_channels, out_channels, time_emb_dim, dropout=0.1):
        super().__init__()
        
        # First conv block
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_channels)
        
        # Time embedding projection
        self.time_mlp = nn.Linear(time_emb_dim, out_channels)
        
        # Second conv block
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_channels)
        
        # Residual connection (if channels change)
        if in_channels != out_channels:
            self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.residual_conv = nn.Identity()
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, t_emb):
        """
        Args:
            x: Input features (batch, in_channels, H, W)
            t_emb: Time embeddings (batch, time_emb_dim)
        
        Returns:
            Output features (batch, out_channels, H, W)
        """
        # Save for residual
        residual = x
        
        # First block
        x = self.conv1(x)
        x = self.norm1(x)
        x = F.silu(x)  # SiLU activation (Swish)
        
        # Add time embedding (broadcast across spatial dimensions)
        t_emb = self.time_mlp(t_emb)
        t_emb = t_emb[:, :, None, None]  # (batch, channels, 1, 1)
        x = x + t_emb
        
        # Second block
        x = self.conv2(x)
        x = self.norm2(x)
        x = self.dropout(x)
        x = F.silu(x)
        
        # Residual connection
        x = x + self.residual_conv(residual)
        
        return x


class AttentionBlock(nn.Module):
    """
    Self-attention block for capturing long-range dependencies.
    Used in U-Net at lower resolutions.
    """
    
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.attention = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=4,
            batch_first=True
        )
    
    def forward(self, x):
        """
        Args:
            x: Input features (batch, channels, H, W)
        
        Returns:
            Output features (batch, channels, H, W)
        """
        batch, channels, h, w = x.shape
        residual = x
        
        # Normalize
        x = self.norm(x)
        
        # Reshape for attention: (batch, h*w, channels)
        x = x.view(batch, channels, h * w).transpose(1, 2)
        
        # Self-attention
        x, _ = self.attention(x, x, x)
        
        # Reshape back: (batch, channels, h, w)
        x = x.transpose(1, 2).view(batch, channels, h, w)
        
        # Residual
        return x + residual


class UNet(nn.Module):
    """
    U-Net architecture for diffusion models.
    
    Architecture:
        - Encoder: Progressively downsample with residual blocks
        - Bottleneck: Attention and residual blocks at lowest resolution
        - Decoder: Progressively upsample with skip connections
        - Time embedding: Injected into all residual blocks
    """
    
    def __init__(self, in_channels=1, out_channels=1, channels=64, time_emb_dim=256):
        super().__init__()
        
        # Time embedding
        self.time_embedding = SinusoidalTimeEmbedding(time_emb_dim // 2)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim // 2, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim)
        )
        
        # Initial convolution
        self.init_conv = nn.Conv2d(in_channels, channels, kernel_size=3, padding=1)
        
        # Encoder (downsampling)
        self.down1 = nn.ModuleList([
            ResidualBlock(channels, channels, time_emb_dim),
            ResidualBlock(channels, channels, time_emb_dim),
        ])
        self.down1_pool = nn.Conv2d(channels, channels, kernel_size=3, stride=2, padding=1)
        
        self.down2 = nn.ModuleList([
            ResidualBlock(channels, channels * 2, time_emb_dim),
            ResidualBlock(channels * 2, channels * 2, time_emb_dim),
        ])
        self.down2_pool = nn.Conv2d(channels * 2, channels * 2, kernel_size=3, stride=2, padding=1)
        
        # Bottleneck
        self.bottleneck = nn.ModuleList([
            ResidualBlock(channels * 2, channels * 2, time_emb_dim),
            AttentionBlock(channels * 2),
            ResidualBlock(channels * 2, channels * 2, time_emb_dim),
        ])
        
        # Decoder (upsampling)
        self.up2 = nn.ModuleList([
            ResidualBlock(channels * 4, channels * 2, time_emb_dim),  # *4 because of skip connection
            ResidualBlock(channels * 2, channels * 2, time_emb_dim),
        ])
        self.up2_upsample = nn.ConvTranspose2d(channels * 2, channels * 2, kernel_size=4, stride=2, padding=1)
        
        self.up1 = nn.ModuleList([
            ResidualBlock(channels * 3, channels, time_emb_dim),  # *3 because of skip connection
            ResidualBlock(channels, channels, time_emb_dim),
        ])
        self.up1_upsample = nn.ConvTranspose2d(channels, channels, kernel_size=4, stride=2, padding=1)
        
        # Final convolution
        self.final_conv = nn.Sequential(
            nn.GroupNorm(8, channels),
            nn.SiLU(),
            nn.Conv2d(channels, out_channels, kernel_size=1)
        )
    
    def forward(self, x, t):
        """
        Args:
            x: Noisy images (batch, in_channels, H, W)
            t: Timesteps (batch,)
        
        Returns:
            Predicted noise (batch, out_channels, H, W)
        """
        # Time embedding
        t_emb = self.time_embedding(t)
        t_emb = self.time_mlp(t_emb)
        
        # Initial conv
        x = self.init_conv(x)
        
        # Encoder
        skip1 = x
        for block in self.down1:
            x = block(x, t_emb)
        x = self.down1_pool(x)
        
        skip2 = x
        for block in self.down2:
            x = block(x, t_emb)
        x = self.down2_pool(x)
        
        # Bottleneck
        for block in self.bottleneck:
            if isinstance(block, AttentionBlock):
                x = block(x)
            else:
                x = block(x, t_emb)
        
        # Decoder
        x = self.up2_upsample(x)
        x = torch.cat([x, skip2], dim=1)  # Skip connection
        for block in self.up2:
            x = block(x, t_emb)
        
        x = self.up1_upsample(x)
        x = torch.cat([x, skip1], dim=1)  # Skip connection
        for block in self.up1:
            x = block(x, t_emb)
        
        # Final conv
        x = self.final_conv(x)
        
        return x


# Create model
model = UNet(in_channels=1, out_channels=1, channels=64, time_emb_dim=256).to(device)

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

# Test forward pass
test_x = torch.randn(4, 1, 28, 28).to(device)
test_t = torch.randint(0, 1000, (4,)).to(device)
test_out = model(test_x, test_t)
print(f"Input shape: {test_x.shape}")
print(f"Output shape: {test_out.shape}")
print(f"Output matches input: {test_out.shape == test_x.shape}")

## 9. Training the Diffusion Model

Now we'll train the model to predict noise.

In [ ]:
def train_epoch(model, dataloader, optimizer, schedule, device):
    """
    Train for one epoch.
    
    Training procedure:
    1. Sample real images x_0
    2. Sample random timesteps t
    3. Sample noise and create noisy images x_t
    4. Predict noise with model
    5. Compute MSE loss between predicted and actual noise
    """
    model.train()
    total_loss = 0
    
    for batch_idx, (x_0, _) in enumerate(tqdm(dataloader, desc="Training", leave=False)):
        x_0 = x_0.to(device)
        batch_size = x_0.shape[0]
        
        # Sample random timesteps
        t = torch.randint(0, schedule.timesteps, (batch_size,), device=device)
        
        # Sample noise and create noisy images
        noise = torch.randn_like(x_0)
        x_t, _ = schedule.forward_diffusion(x_0, t, noise)
        
        # Predict noise
        predicted_noise = model(x_t, t)
        
        # Compute loss
        loss = F.mse_loss(predicted_noise, noise)
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)


# Training configuration
num_epochs = 10
learning_rate = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print(f"Training configuration:")
print(f"  Epochs: {num_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Batch size: {batch_size}")
print(f"  Timesteps: {timesteps}")

Run the training loop with periodic sampling to visualize progress.

In [ ]:
# Training loop
losses = []

print("Training diffusion model...\n")

for epoch in range(num_epochs):
    loss = train_epoch(model, train_loader, optimizer, schedule, device)
    losses.append(loss)
    
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {loss:.6f}")
    
    # Visualize samples every few epochs
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print("Generating samples...")
        # We'll implement sampling in the next section

print("\nTraining complete!")

# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. DDPM Sampling - Generating Images

Now let's implement the reverse process to generate images from noise.

In [ ]:
@torch.no_grad()
def sample_ddpm(model, schedule, image_shape, device, num_samples=16):
    """
    Sample images using DDPM algorithm.
    
    Algorithm:
    1. Start with pure noise x_T ~ N(0, I)
    2. For t = T down to 1:
        - Predict noise with model
        - Compute predicted x_0
        - Sample x_{t-1} from posterior
    3. Return x_0
    
    Args:
        model: Trained U-Net
        schedule: Diffusion schedule
        image_shape: Shape of images (C, H, W)
        device: Device
        num_samples: Number of images to generate
    
    Returns:
        Generated images (num_samples, C, H, W)
    """
    model.eval()
    
    # Start with pure noise
    x = torch.randn(num_samples, *image_shape, device=device)
    
    # Reverse process
    for t in tqdm(reversed(range(schedule.timesteps)), desc="Sampling", total=schedule.timesteps):
        # Current timestep
        t_batch = torch.full((num_samples,), t, device=device, dtype=torch.long)
        
        # Predict noise
        predicted_noise = model(x, t_batch)
        
        # Get coefficients
        alpha_t = schedule.alphas[t]
        alpha_bar_t = schedule.alphas_cumprod[t]
        beta_t = schedule.betas[t]
        
        # Predict x_0 from x_t and predicted noise
        # x_0 = (x_t - sqrt(1 - alpha_bar_t) * noise) / sqrt(alpha_bar_t)
        predicted_x0 = (
            x - torch.sqrt(1 - alpha_bar_t) * predicted_noise
        ) / torch.sqrt(alpha_bar_t)
        
        # Clip to valid range
        predicted_x0 = torch.clamp(predicted_x0, -1, 1)
        
        if t > 0:
            # Compute mean of posterior q(x_{t-1} | x_t, x_0)
            alpha_bar_t_prev = schedule.alphas_cumprod[t - 1]
            
            # Mean of posterior
            posterior_mean = (
                torch.sqrt(alpha_bar_t_prev) * beta_t / (1 - alpha_bar_t) * predicted_x0 +
                torch.sqrt(alpha_t) * (1 - alpha_bar_t_prev) / (1 - alpha_bar_t) * x
            )
            
            # Variance of posterior
            posterior_variance = schedule.posterior_variance[t]
            
            # Sample x_{t-1}
            noise = torch.randn_like(x)
            x = posterior_mean + torch.sqrt(posterior_variance) * noise
        else:
            # Final step: no noise
            x = predicted_x0
    
    return x


# Generate samples
print("Generating images with DDPM...")
generated_images = sample_ddpm(model, schedule, image_shape=(1, 28, 28), device=device, num_samples=64)

show_images(generated_images, "Generated Images (DDPM)", nrow=8)

## 11. DDIM Sampling - Faster Generation

**Problem with DDPM**: Requires many steps (e.g., 1000) for good quality

**DDIM (Denoising Diffusion Implicit Models)** enables:
- **Fewer steps**: Can skip timesteps (e.g., use only 50 steps)
- **Deterministic sampling**: Can be made deterministic (no random noise)
- **Same model**: Uses the same trained DDPM model!

### Key Difference:

DDIM uses a different reverse process that allows skipping steps:

$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \underbrace{\left(\frac{x_t - \sqrt{1-\bar{\alpha}_t} \epsilon_\theta(x_t, t)}{\sqrt{\bar{\alpha}_t}}\right)}_{\text{predicted } x_0} + \sqrt{1-\bar{\alpha}_{t-1} - \sigma_t^2} \epsilon_\theta(x_t, t) + \sigma_t \epsilon$$

Where $\sigma_t$ controls stochasticity:
- $\sigma_t = 0$: Deterministic (DDIM)
- $\sigma_t = \sqrt{\beta_t}$: Stochastic (DDPM)

In [ ]:
@torch.no_grad()
def sample_ddim(model, schedule, image_shape, device, num_samples=16, num_steps=50, eta=0.0):
    """
    Sample images using DDIM algorithm.
    
    DDIM allows using fewer steps than DDPM by skipping timesteps.
    
    Args:
        model: Trained U-Net
        schedule: Diffusion schedule
        image_shape: Shape of images (C, H, W)
        device: Device
        num_samples: Number of images to generate
        num_steps: Number of denoising steps (can be less than schedule.timesteps)
        eta: Stochasticity parameter (0 = deterministic, 1 = stochastic like DDPM)
    
    Returns:
        Generated images (num_samples, C, H, W)
    """
    model.eval()
    
    # Create subsequence of timesteps
    step_size = schedule.timesteps // num_steps
    timesteps = list(range(0, schedule.timesteps, step_size))
    timesteps = list(reversed(timesteps))
    
    # Start with pure noise
    x = torch.randn(num_samples, *image_shape, device=device)
    
    # Reverse process
    for i, t in enumerate(tqdm(timesteps, desc="DDIM Sampling")):
        # Current timestep
        t_batch = torch.full((num_samples,), t, device=device, dtype=torch.long)
        
        # Predict noise
        predicted_noise = model(x, t_batch)
        
        # Get alpha values
        alpha_bar_t = schedule.alphas_cumprod[t]
        
        # Predict x_0
        predicted_x0 = (
            x - torch.sqrt(1 - alpha_bar_t) * predicted_noise
        ) / torch.sqrt(alpha_bar_t)
        predicted_x0 = torch.clamp(predicted_x0, -1, 1)
        
        if i < len(timesteps) - 1:
            # Get next timestep
            t_prev = timesteps[i + 1]
            alpha_bar_t_prev = schedule.alphas_cumprod[t_prev]
            
            # Compute sigma
            sigma = eta * torch.sqrt(
                (1 - alpha_bar_t_prev) / (1 - alpha_bar_t) * (1 - alpha_bar_t / alpha_bar_t_prev)
            )
            
            # Compute predicted x_{t-1}
            pred_dir = torch.sqrt(1 - alpha_bar_t_prev - sigma**2) * predicted_noise
            x = torch.sqrt(alpha_bar_t_prev) * predicted_x0 + pred_dir
            
            if sigma > 0:
                noise = torch.randn_like(x)
                x = x + sigma * noise
        else:
            # Final step
            x = predicted_x0
    
    return x


# Generate samples with DDIM (50 steps instead of 1000)
print("Generating images with DDIM (50 steps)...")
generated_images_ddim = sample_ddim(
    model, schedule, image_shape=(1, 28, 28), device=device, 
    num_samples=64, num_steps=50, eta=0.0
)

show_images(generated_images_ddim, "Generated Images (DDIM, 50 steps)", nrow=8)

Compare sampling speed between DDPM and DDIM.

In [ ]:
import time

# Compare sampling times
print("Comparing DDPM vs DDIM sampling speed...\n")

# DDPM (1000 steps)
start = time.time()
_ = sample_ddpm(model, schedule, image_shape=(1, 28, 28), device=device, num_samples=16)
ddpm_time = time.time() - start

# DDIM (50 steps)
start = time.time()
_ = sample_ddim(model, schedule, image_shape=(1, 28, 28), device=device, num_samples=16, num_steps=50)
ddim_time = time.time() - start

print(f"DDPM (1000 steps): {ddpm_time:.2f}s")
print(f"DDIM (50 steps): {ddim_time:.2f}s")
print(f"Speedup: {ddpm_time / ddim_time:.1f}x")

## 12. Visualizing the Denoising Process

Let's visualize how an image is gradually denoised during sampling.

In [ ]:
@torch.no_grad()
def visualize_denoising(model, schedule, device, num_steps=50):
    """
    Visualize the denoising process step by step.
    """
    model.eval()
    
    # Timesteps to visualize
    step_size = schedule.timesteps // num_steps
    timesteps = list(range(0, schedule.timesteps, step_size))
    timesteps = list(reversed(timesteps))
    
    # Steps to save
    save_steps = [0, 10, 20, 30, 40, 49]
    saved_images = []
    
    # Start with noise
    x = torch.randn(1, 1, 28, 28, device=device)
    
    for i, t in enumerate(timesteps):
        if i in save_steps:
            saved_images.append(x.clone())
        
        # Denoise
        t_batch = torch.tensor([t], device=device)
        predicted_noise = model(x, t_batch)
        
        alpha_bar_t = schedule.alphas_cumprod[t]
        predicted_x0 = (x - torch.sqrt(1 - alpha_bar_t) * predicted_noise) / torch.sqrt(alpha_bar_t)
        predicted_x0 = torch.clamp(predicted_x0, -1, 1)
        
        if i < len(timesteps) - 1:
            t_prev = timesteps[i + 1]
            alpha_bar_t_prev = schedule.alphas_cumprod[t_prev]
            pred_dir = torch.sqrt(1 - alpha_bar_t_prev) * predicted_noise
            x = torch.sqrt(alpha_bar_t_prev) * predicted_x0 + pred_dir
        else:
            x = predicted_x0
    
    saved_images.append(x.clone())
    return torch.cat(saved_images, dim=0)


# Visualize denoising for multiple samples
fig, axes = plt.subplots(4, 7, figsize=(14, 8))

for row in range(4):
    images = visualize_denoising(model, schedule, device, num_steps=50)
    
    for col in range(7):
        img = (images[col] + 1) / 2  # Denormalize
        img = torch.clamp(img, 0, 1)
        axes[row, col].imshow(img.squeeze().cpu().numpy(), cmap='gray')
        axes[row, col].axis('off')
        if row == 0:
            step = [0, 10, 20, 30, 40, 49, 50][col]
            axes[row, col].set_title(f'Step {step}', fontsize=10)

plt.suptitle('Denoising Process: From Noise to Image', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Key Observations:

1. **Early steps**: Remove large-scale noise, establish rough structure
2. **Middle steps**: Refine shapes and features
3. **Late steps**: Add fine details and sharpness
4. **Hierarchical generation**: Coarse-to-fine, like how humans draw!

## 13. Connection to Score-Based Models

Diffusion models are deeply connected to **score-based generative models**.

### What is the Score?

The **score** is the gradient of the log probability density:

$$s_\theta(x, t) = \nabla_x \log p_t(x)$$

**Intuition**: Points in the direction of higher probability (uphill in probability space)

### Key Insight:

The noise prediction network is actually learning the score!

$$\epsilon_\theta(x_t, t) \approx -\sqrt{1 - \bar{\alpha}_t} \nabla_{x_t} \log p(x_t)$$

In other words:
- **Noise prediction** $\leftrightarrow$ **Score estimation**
- Removing noise $\leftrightarrow$ Following the score (moving to higher probability)

### Why This Matters:

1. **Theoretical foundation**: Score matching provides rigorous mathematical basis
2. **Alternative training**: Can train with score matching instead of noise prediction
3. **Unified view**: Connects diffusion models to other generative approaches
4. **Extensions**: Enables methods like score-based SDEs (stochastic differential equations)

### Langevin Dynamics:

The reverse process can be seen as **Langevin dynamics**:

$$x_{t-1} = x_t + \frac{\delta}{2} \nabla_x \log p(x_t) + \sqrt{\delta} z$$

This is a classical physics algorithm for sampling from distributions!

## 14. Comparing Noise Schedules

The noise schedule $\beta_t$ significantly affects generation quality.

**Common schedules:**
1. **Linear**: $\beta_t$ increases linearly (what we used)
2. **Cosine**: Slower at start, based on cosine function
3. **Quadratic**: $\beta_t$ increases quadratically

Let's visualize different schedules.

In [ ]:
def cosine_beta_schedule(timesteps, s=0.008):
    """
    Cosine schedule as proposed in Improved DDPM paper.
    """
    steps = timesteps + 1
    t = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((t / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.9999)


# Compare schedules
timesteps = 1000
linear_betas = torch.linspace(0.0001, 0.02, timesteps)
cosine_betas = cosine_beta_schedule(timesteps)

# Compute alpha_bar for both
linear_alphas_cumprod = torch.cumprod(1 - linear_betas, dim=0)
cosine_alphas_cumprod = torch.cumprod(1 - cosine_betas, dim=0)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Beta schedules
axes[0].plot(linear_betas.numpy(), label='Linear', linewidth=2)
axes[0].plot(cosine_betas.numpy(), label='Cosine', linewidth=2)
axes[0].set_xlabel('Timestep')
axes[0].set_ylabel(r'$\beta_t$')
axes[0].set_title('Noise Schedules')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Alpha_bar (signal remaining)
axes[1].plot(linear_alphas_cumprod.numpy(), label='Linear', linewidth=2)
axes[1].plot(cosine_alphas_cumprod.numpy(), label='Cosine', linewidth=2)
axes[1].set_xlabel('Timestep')
axes[1].set_ylabel(r'$\bar{\alpha}_t$ (signal remaining)')
axes[1].set_title('Signal Preservation Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Schedule comparison:")
print(f"Linear - Final alpha_bar: {linear_alphas_cumprod[-1]:.6f}")
print(f"Cosine - Final alpha_bar: {cosine_alphas_cumprod[-1]:.6f}")
print("\nCosine schedule preserves more signal early, useful for high-resolution images.")

## 15. Latent Space Interpolation

Unlike VAEs, diffusion models don't have an explicit latent space. However, we can interpolate in the **noise space**.

In [ ]:
@torch.no_grad()
def interpolate_noise(model, schedule, noise1, noise2, num_steps=8, sampling_steps=50):
    """
    Interpolate between two noise vectors and generate images.
    """
    model.eval()
    
    # Create interpolation
    alphas = torch.linspace(0, 1, num_steps, device=device)
    interpolated_noises = []
    
    for alpha in alphas:
        noise_interp = (1 - alpha) * noise1 + alpha * noise2
        interpolated_noises.append(noise_interp)
    
    interpolated_noises = torch.cat(interpolated_noises, dim=0)
    
    # Generate images from interpolated noises using DDIM
    step_size = schedule.timesteps // sampling_steps
    timesteps = list(range(0, schedule.timesteps, step_size))
    timesteps = list(reversed(timesteps))
    
    x = interpolated_noises
    
    for i, t in enumerate(tqdm(timesteps, desc="Generating", leave=False)):
        t_batch = torch.full((num_steps,), t, device=device, dtype=torch.long)
        predicted_noise = model(x, t_batch)
        
        alpha_bar_t = schedule.alphas_cumprod[t]
        predicted_x0 = (x - torch.sqrt(1 - alpha_bar_t) * predicted_noise) / torch.sqrt(alpha_bar_t)
        predicted_x0 = torch.clamp(predicted_x0, -1, 1)
        
        if i < len(timesteps) - 1:
            t_prev = timesteps[i + 1]
            alpha_bar_t_prev = schedule.alphas_cumprod[t_prev]
            pred_dir = torch.sqrt(1 - alpha_bar_t_prev) * predicted_noise
            x = torch.sqrt(alpha_bar_t_prev) * predicted_x0 + pred_dir
        else:
            x = predicted_x0
    
    return x


# Create two random noise vectors
noise1 = torch.randn(1, 1, 28, 28, device=device)
noise2 = torch.randn(1, 1, 28, 28, device=device)

# Interpolate
print("Interpolating in noise space...")
interpolated = interpolate_noise(model, schedule, noise1, noise2, num_steps=12, sampling_steps=50)

show_images(interpolated, "Noise Space Interpolation", nrow=12)

## 16. Key Takeaways

### What We Learned:

#### 1. Forward Diffusion Process
- Gradually adds Gaussian noise over $T$ timesteps
- Controlled by noise schedule $\beta_t$
- Can jump to any timestep using closed-form: $x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon$
- Eventually transforms any data into pure noise

#### 2. Reverse Diffusion Process
- Learns to remove noise step by step
- Train network to predict noise: $\epsilon_\theta(x_t, t)$
- Simple MSE loss: $\|\epsilon - \epsilon_\theta(x_t, t)\|^2$
- Generate by reversing: noise → image

#### 3. U-Net with Time Embeddings
- Standard architecture for diffusion models
- Sinusoidal time embeddings encode timestep information
- Skip connections preserve spatial details
- Attention blocks capture long-range dependencies

#### 4. Sampling Strategies
- **DDPM**: Original algorithm, requires all $T$ steps
- **DDIM**: Faster sampling by skipping steps
- **Trade-off**: Speed vs quality (fewer steps = faster but potentially lower quality)
- **Deterministic option**: DDIM with $\eta=0$

#### 5. Connection to Score-Based Models
- Noise prediction ≈ Score estimation
- Denoising ≈ Following probability gradient
- Theoretical foundation in Langevin dynamics

### Why Diffusion Models Are Powerful:

**Advantages:**
1. **Stable training**: Unlike GANs, no adversarial training
2. **High quality**: Sharp, detailed images
3. **Mode coverage**: Generates diverse samples, no mode collapse
4. **Scalability**: Works well at high resolutions
5. **Flexibility**: Easy to condition on text, class, etc.

**Limitations:**
1. **Slow sampling**: Many denoising steps needed (DDIM helps)
2. **Training cost**: Requires substantial compute
3. **Memory**: Large models for high resolution

### Real-World Applications:

1. **Text-to-Image**: Stable Diffusion, DALL-E 2, Midjourney
2. **Image Editing**: Inpainting, outpainting, style transfer
3. **Super-Resolution**: Upscaling images
4. **Video Generation**: Extending to temporal dimension
5. **3D Generation**: Generating 3D shapes and scenes
6. **Audio**: Generating music and speech

### Comparison Summary:

| Aspect | VAE | GAN | Diffusion |
|--------|-----|-----|----------|
| **Quality** | Blurry | Sharp | Sharp |
| **Training** | Stable | Unstable | Stable |
| **Speed** | Fast (1 step) | Fast (1 step) | Slow (many steps) |
| **Diversity** | Good | Mode collapse risk | Excellent |
| **Likelihood** | Tractable | Intractable | Tractable |
| **Use Case** | Compression | High-quality images | State-of-the-art generation |

## 17. Extensions and Advanced Topics

**Topics to explore further:**

### 1. Conditional Diffusion Models
- Add class labels or text embeddings
- Classifier guidance: Use classifier gradients to guide generation
- Classifier-free guidance: Train conditional and unconditional models together

### 2. Latent Diffusion Models (Stable Diffusion)
- Run diffusion in compressed latent space (using VAE encoder/decoder)
- Much faster and more memory efficient
- Enables high-resolution generation (512×512, 1024×1024)

### 3. Improved Architectures
- Improved U-Net: Better attention, normalization
- DiT (Diffusion Transformer): Replace U-Net with Transformer
- Cascaded models: Multiple diffusion models at different resolutions

### 4. Better Sampling
- DPM-Solver: Even faster sampling with better quality
- Guidance schedules: Dynamic classifier-free guidance
- Progressive distillation: Train faster models from slow ones

### 5. Score-Based SDEs
- Continuous-time diffusion using stochastic differential equations
- Probability flow ODEs for deterministic sampling
- More flexible noise schedules

### 6. Applications
- Image inpainting and editing
- Text-to-image generation (CLIP conditioning)
- Video generation
- 3D shape generation

## 18. Exercises and Experiments

**Try these experiments to deepen understanding:**

### Experiment 1: Different Noise Schedules
- Train models with linear vs cosine schedules
- Compare generation quality and training dynamics
- Try very aggressive or gentle schedules

### Experiment 2: Varying Timesteps
- Train with $T = 100$, $T = 500$, $T = 2000$
- How does this affect quality and training time?
- Is there a sweet spot?

### Experiment 3: Architecture Ablations
- Remove attention blocks - what happens?
- Remove skip connections - what happens?
- Try different numbers of channels

### Experiment 4: DDIM Steps
- Generate with 10, 25, 50, 100, 250 steps
- Plot quality vs speed trade-off
- Find minimum steps for acceptable quality

### Experiment 5: Other Datasets
- Try Fashion-MNIST, CIFAR-10
- What changes are needed for color images?
- How does complexity affect training?

### Experiment 6: Conditional Generation
- Add class conditioning to the model
- Implement classifier-free guidance
- Generate specific digit classes

## 19. Further Reading

**Foundational Papers:**

1. **DDPM**: "Denoising Diffusion Probabilistic Models" (Ho et al., 2020)
   - Original diffusion model paper
   - Introduced simple noise prediction objective

2. **DDIM**: "Denoising Diffusion Implicit Models" (Song et al., 2020)
   - Faster sampling via skipping steps
   - Deterministic generation

3. **Score-Based**: "Score-Based Generative Modeling through SDEs" (Song et al., 2021)
   - Continuous-time formulation
   - Unified framework

4. **Improved DDPM**: "Improved Denoising Diffusion Probabilistic Models" (Nichol & Dhariwal, 2021)
   - Better noise schedules
   - Learned variances

5. **Classifier Guidance**: "Diffusion Models Beat GANs" (Dhariwal & Nichol, 2021)
   - Conditional generation
   - Guidance for better quality

6. **Latent Diffusion**: "High-Resolution Image Synthesis with Latent Diffusion Models" (Rombach et al., 2022)
   - Stable Diffusion architecture
   - Diffusion in latent space

**Tutorials and Resources:**
- Lilian Weng's blog: "What are Diffusion Models?"
- Hugging Face Diffusers library documentation
- Annotated Diffusion Model tutorial

**Modern Applications:**
- Stable Diffusion: Open-source text-to-image
- DALL-E 2: OpenAI's text-to-image
- Imagen: Google's text-to-image
- Midjourney: Commercial image generation